In [1]:
#undef __noinline__

# Introduction to CUDA

This notebook is going to show you beginner friendly examples of CUDA syntax. 

Starting with declaring  a __global__ kernel that runs on the GPU and adds two numbers

In [2]:
__global__ void add( int a, int b, int *c ) {
 *c = a + b;
}

Next we have to run this kernel. The `<<< >>>` syntax helps us with that. The first number is how many blocks and the second is how many threads we are going to use.

Before everything we need to allocate vram with cudaMalloc() on the gpu to store the result of both added numbers. Then we run the kernel with 2 and 7.

Next we need to copy that vram memory to the host. We do that with cudaMemcpy(). And finally we print the final number and free the memory

In [3]:
#include <iostream>

int c;
int *d_c;

cudaMalloc( (void**)&d_c, sizeof(int) );

add<<<1,1>>>( 2, 7, d_c );

 cudaMemcpy( &c, d_c, sizeof(int), cudaMemcpyDeviceToHost );

 printf( "2 + 7 = %d\n", c );
 cudaFree( d_c );
 

2 + 7 = 9


Next we are going to try something a little bit more difficult. We are going to add two vectors index by index and store the result in a third vector.

Before any code you can see a visualization of the addition.

In [4]:
#include "headers/display.hpp"

show_image("assets/vectoradd.png", 25);

First we need the global function. As you can see the kernel performs only one addition. In order to add the whole vector we need multiple blocks running at the same time.

In [5]:
const int N = 20;

__global__ void vector_add( int *a, int *b, int *c ) {
     int tid = blockIdx.x; // handle the data at this index
     if (tid < N)
     c[tid] = a[tid] + b[tid];
}

Parallel programming is what makes the GPU great for a big number of the same operations being performed simultaneously on different pieces of data.

We can see that we call an **N** number of kernels to preform only one addition in the vector each. In this example we don`t save that much time but parallel programming works in our favour because when we go to in to the millions of operations we save a lot of time instead of adding them one by one with the CPU.

Bellow you can see how blocks and threads are spaced out in the grid.

In [6]:
show_image("assets/CudaGrid.png", 50);

In [7]:
int v_a[N], v_b[N], v_c[N];
int *dev_a, *dev_b, *dev_c;

// allocate the memory on the GPU
cudaMalloc( (void**)&dev_a, N * sizeof(int) );
cudaMalloc( (void**)&dev_b, N * sizeof(int) );
cudaMalloc( (void**)&dev_c, N * sizeof(int) );

// fill the arrays 'a' and 'b' on the CPU
for (int i=0; i<N; i++) {
    v_a[i] = i;
    v_b[i] = i * i;
}

// copy the arrays 'a' and 'b' to the GPU
cudaMemcpy( dev_a, v_a, N * sizeof(int), cudaMemcpyHostToDevice );
cudaMemcpy( dev_b, v_b, N * sizeof(int), cudaMemcpyHostToDevice );

vector_add<<<N,1>>>( dev_a, dev_b, dev_c );

// copy the array 'c' back from the GPU to the CPU
cudaMemcpy( v_c, dev_c, N * sizeof(int), cudaMemcpyDeviceToHost );

// display the results
for (int i=0; i<N; i++) {
    printf( "%d + %d = %d\n", v_a[i], v_b[i], v_c[i] );
}

// free the memory allocated on the GPU
cudaFree( dev_a );
cudaFree( dev_b );
cudaFree( dev_c );

0 + 0 = 0
1 + 1 = 2
2 + 4 = 6
3 + 9 = 12
4 + 16 = 20
5 + 25 = 30
6 + 36 = 42
7 + 49 = 56
8 + 64 = 72
9 + 81 = 90
10 + 100 = 110
11 + 121 = 132
12 + 144 = 156
13 + 169 = 182
14 + 196 = 210
15 + 225 = 240
16 + 256 = 272
17 + 289 = 306
18 + 324 = 342
19 + 361 = 380
